# H.U.G.H. Voice Clone — Skarsgard Profile
Fine-tunes XTTS v2 on Skarsgard/Cerdic reference audio.
Runtime: T4 GPU. Set Runtime > Change runtime type > T4 GPU before running.

In [ ]:
!pip install -q TTS==0.22.0 pydub faster-whisper soundfile
!apt-get -qq install -y ffmpeg
print('Dependencies installed')

In [ ]:
from google.colab import files
import os
os.makedirs('/content/ref', exist_ok=True)
os.makedirs('/content/train', exist_ok=True)
os.makedirs('/content/out', exist_ok=True)
print('Upload the 4 Skarsgard MP3s from voice_raw/')
uploaded = files.upload()
for fn in uploaded:
    with open(f'/content/ref/{fn}', 'wb') as f:
        f.write(uploaded[fn])
print(f'{len(uploaded)} files uploaded')

In [ ]:
from pydub import AudioSegment
from pydub.silence import detect_silence
import glob

all_segs = []
for mp3 in sorted(glob.glob('/content/ref/*.mp3')):
    name = os.path.splitext(os.path.basename(mp3))[0]
    audio = AudioSegment.from_file(mp3).set_channels(1).set_frame_rate(22050)
    audio = audio.apply_gain(-20 - audio.dBFS)
    silences = detect_silence(audio, min_silence_len=300, silence_thresh=-35)
    breaks = [(s+e)//2 for s,e in silences]
    cursor, segs = 0, []
    for b in breaks:
        if b - cursor >= 3000:
            if b - cursor <= 12000:
                segs.append(audio[cursor:b])
                cursor = b
    if len(audio) - cursor >= 3000:
        segs.append(audio[cursor:])
    for i, seg in enumerate(segs):
        p = f'/content/train/{name}_seg{i:03d}.wav'
        seg.export(p, format='wav')
        all_segs.append(p)
    print(f'{name}: {len(segs)} segments ({len(audio)/1000:.1f}s)')
print(f'{len(all_segs)} total segments')

In [ ]:
from faster_whisper import WhisperModel
import json
model = WhisperModel('base.en', device='cuda', compute_type='float16')
meta = []
for wav in sorted(all_segs):
    segs, _ = model.transcribe(wav, language='en')
    text = ' '.join([s.text.strip() for s in segs])
    if len(text) > 5:
        meta.append({'audio_file': wav, 'text': text, 'language': 'en'})
        print(f'{os.path.basename(wav)}: {text[:80]}')
with open('/content/train/metadata.json','w') as f:
    json.dump(meta, f, indent=2)
print(f'{len(meta)} segments transcribed')
del model
import torch; torch.cuda.empty_cache()

In [ ]:
# Zero-shot clone test first (quick sanity check)
from TTS.api import TTS
from IPython.display import Audio, display
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
ref = sorted(all_segs)[0]
tests = [
    'Workshop online. Clifford field nominal, substrate warm. What wisdom do you seek today?',
    'Copy that Grizz. Running diagnostics on the Proxmox cluster now. Stand by for telemetry.',
    'Soul anchor verified. Integrity hash matches. The Workshop is open.',
]
for i, t in enumerate(tests):
    p = f'/content/out/zeroshot_{i}.wav'
    tts.tts_to_file(text=t, speaker_wav=ref, language='en', file_path=p)
    print(f'Sample {i+1}: {t[:60]}')
    display(Audio(p))
print('Listen above. If voice is close enough, skip to audiobook cell.')

In [ ]:
# Optional: Fine-tune for better quality (20-30 min on T4)
# Only run this if zero-shot quality is not good enough
import json
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

with open('/content/train/metadata.json') as f:
    meta = json.load(f)

# Write csvs
with open('/content/train/train.csv','w') as f:
    for m in meta[:-2]:
        f.write(f'{m["audio_file"]}|{m["text"]}|{m["language"]}\n')
with open('/content/train/eval.csv','w') as f:
    for m in meta[-2:]:
        f.write(f'{m["audio_file"]}|{m["text"]}|{m["language"]}\n')

print(f'Train: {len(meta)-2}, Eval: 2')
print('Starting fine-tune... this takes 20-30 min on T4')

In [ ]:
# Generate audiobook MP3
import re
from pydub import AudioSegment as AS
from google.colab import files as gfiles

# Upload the PEAR-OS markdown file
print('Upload PEAR-OS-Apple-Ecosystem-Research.md')
up = gfiles.upload()
text = list(up.values())[0].decode('utf-8')
# Strip markdown formatting
text = re.sub(r'#+ ', '', text)
text = re.sub(r'\*\*|__|\*|_', '', text)
text = re.sub(r'```[\s\S]*?```', '', text)
text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

# Split into chunks
sents = re.split(r'(?<=[.!?])\s+', text.strip())
chunks, cur = [], ''
for s in sents:
    if len(cur) + len(s) > 250 and cur:
        chunks.append(cur.strip())
        cur = s
    else:
        cur = cur + ' ' + s if cur else s
if cur.strip(): chunks.append(cur.strip())

print(f'Generating {len(chunks)} audio chunks...')
full = AS.empty()
pause = AS.silent(duration=600)
ref = sorted(all_segs)[0]
for i, c in enumerate(chunks):
    if not c.strip(): continue
    p = f'/content/out/book_{i:04d}.wav'
    tts.tts_to_file(text=c, speaker_wav=ref, language='en', file_path=p)
    full += AS.from_wav(p) + pause
    if i % 5 == 0: print(f'  {i+1}/{len(chunks)}')

mp3 = '/content/out/HUGH_PEAR_OS_Audiobook.mp3'
full.export(mp3, format='mp3', bitrate='192k',
    tags={'artist':'H.U.G.H.','album':'Grizzly Medicine','title':'PEAR-OS Research'})
print(f'Audiobook: {len(full)/1000:.0f}s')
gfiles.download(mp3)